# LecGap Phase 3 — Fine-tune prerequisite classifier (GPU)

This notebook fine-tunes a cross-encoder transformer over **LectureBank 1.0**
prerequisite pairs and evaluates it with the same nested 5-fold CV used locally.

**Input (Kaggle dataset):** two CSVs are expected at
`/kaggle/input/datasets/ayushdevadiga/lecturebank/`
    - `prerequisite_annotation.csv` — `(Source_Topic_ID, Target_Topic_ID, If_prerequisite)`
    - `208topics.csv` — `(id, Topic, Topic_Link)`
If your dataset lives at a different path, update `INPUT_DIR` in the training cell below.

**Output:** the fine-tuned model is written to `/kaggle/working/model/` — download
it (the `model/` folder) and load it on CPU for inference in the LecGap pipeline.

Set **Accelerator = GPU T4** and **Internet = On** (Internet is only needed to
download the pretrained MiniLM checkpoint; the notebook code itself is embedded).

In [14]:
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

GPU available: True
GPU: Tesla T4


In [15]:
!pip install -q transformers sentence-transformers datasets scikit-learn

### 1. Core module — training/export helpers

This cell writes `backend/pipeline/fine_tune.py` to disk (so the subprocess can import it) and loads it.

In [16]:
import os
os.makedirs('/kaggle/working/backend/pipeline', exist_ok=True)
fine_tune_src = (r'''"""
Phase 3 — fine-tuning the pretrained encoder on LectureBank pairs.

Builds on the frozen-encoder baseline (classify_prerequisites.py) by actually
training the transformer over labeled (A, B) pairs, per plan/EVALUATION.md
"fine-tune a pretrained embedding model (not train from scratch)".

Design:
  * Cross-encoder style: loads the MiniLM checkpoint and adds a binary
    sequence-classification head, then fine-tunes ALL weights over the pairs.
    This is trained with a plain PyTorch loop (no trainer/datasets coupling) so
    the exact same code runs locally on CPU and on Kaggle GPU.
  * Paired texts are fed as "[A] [SEP] [B]" so the model attends jointly, i.e.
    it can answer "is A a prerequisite of B?" — the ordering matters and the
    model sees it.
  * Deterministic seed for reproducible CV.
"""

from typing import List, Optional, Sequence, Tuple

import numpy as np

_DEF_BASE = "sentence-transformers/all-MiniLM-L6-v2"


def build_train_triples(
    pairs: Sequence[Tuple[str, str]],
    labels: Sequence[int],
    *,
    max_neg_ratio: int = 8,
    random_state: Optional[int] = None,
) -> List[Tuple[str, str, int]]:
    """Undersample negatives to max_neg_ratio:1 vs positives.

    Returns [(text_a, text_b, label), ...] ready for training.
    """
    pos = [(p[0], p[1], 1) for p, l in zip(pairs, labels) if l == 1]
    neg = [(p[0], p[1], 0) for p, l in zip(pairs, labels) if l == 0]
    import random as _random

    rng = _random.Random(random_state)
    neg = rng.sample(neg, min(len(neg), len(pos) * max_neg_ratio))
    all_ = pos + neg
    rng.shuffle(all_)
    return all_


def _pair_text(a: str, b: str) -> str:
    """Format a pair for the cross-encoder. Order is meaningful (A precedes B)."""
    return f"{a} [SEP] {b}"


def _build_model(base_model: str, device: str):
    import torch
    from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(base_model)
    # Convert the ST/MPNet checkpoint into a binary sequence-classification
    # head, keeping its pretrained weights (they are loaded as the base).
    config = AutoConfig.from_pretrained(base_model, num_labels=1)
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model, config=config, ignore_mismatched_sizes=True
    )
    model.to(device)
    return model, tokenizer


def fine_tune_cross_encoder(
    train_triples: Sequence[Tuple[str, str, int]],
    *,
    base_model: str = _DEF_BASE,
    val_triples: Optional[Sequence[Tuple[str, str, int]]] = None,
    epochs: int = 3,
    batch_size: int = 32,
    lr: float = 2e-5,
    seed: int = 42,
    device: str = None,
):
    """Fine-tune a (cross-encoder) transformer on (a, b, label) triples.

    Returns (model, tokenizer) with the trained weights. The full network
    (transformer body + classification head) is trained — this is the
    "fine-tune, don't train from scratch" step the plan calls for.
    """
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    from torch.optim import AdamW
    from torch.nn import BCEWithLogitsLoss

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    torch.manual_seed(seed)
    np.random.seed(seed)

    model, tokenizer = _build_model(base_model, device)

    # Encode all texts once.
    texts = [ _pair_text(a, b) for a, b, _ in train_triples ]
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt",
    )
    labels = torch.tensor([float(l) for _, _, l in train_triples], dtype=torch.float)
    dataset = TensorDataset(enc["input_ids"], enc["attention_mask"], labels)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    optimizer = AdamW(model.parameters(), lr=lr)
    loss_fn = BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        total = 0.0
        for step, (ids, mask, lbl) in enumerate(loader):
            ids, mask, lbl = ids.to(device), mask.to(device), lbl.to(device)
            optimizer.zero_grad()
            logits = model(input_ids=ids, attention_mask=mask, labels=None).logits
            loss = loss_fn(logits.squeeze(-1), lbl)
            loss.backward()
            optimizer.step()
            total += loss.item()
        avg = total / max(1, len(loader))
        # (val logging intentionally omitted — eval is done by evaluate_classifier.)

    return model, tokenizer


def export_model(model, tokenizer, output_dir: str) -> str:
    """Save the fine-tuned transformer + tokenizer; return the path."""
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    return output_dir


def load_model(model_dir: str, device: str = None):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    return model, tokenizer


def predict_pairs(model, tokenizer, pairs: Sequence[Tuple[str, str]]) -> np.ndarray:
    """Return a per-pair logit; higher = A is more likely a prereq of B."""
    import torch

    texts = [_pair_text(a, b) for a, b in pairs]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    device = next(model.parameters()).device
    enc = {k: v.to(device) for k, v in enc.items()}
    if len(texts) == 1:
        for k in enc:
            enc[k] = enc[k].unsqueeze(0)
    with torch.no_grad():
        logits = model(**enc).logits
    return logits.squeeze(-1).detach().cpu().numpy()
''')
with open('/kaggle/working/backend/pipeline/fine_tune.py', 'w') as _f:
    _f.write(fine_tune_src)
print('wrote backend/pipeline/fine_tune.py')


wrote backend/pipeline/fine_tune.py


### 2. Training + evaluation script

This cell writes `scripts/kaggle_fine_tune.py` (nested 5-fold CV plus final model export)
to `/kaggle/working/`.

In [17]:
kaggle_src = r'''"""
Phase 3 — fine-tune the encoder and evaluate 5-fold CV, GPU-or-CPU agnostic.

This is the training+benchmark counterpart to evaluate_classifier.py (which
scores the frozen-encoder baseline). It fine-tunes a cross-encoder transformer
over LectureBank pairs and reports precision/recall/F1 with the SAME nested
threshold-selection methodology (threshold picked on a held-out val slice,
never the test fold).

Runs identically on:
  * Local CPU:  & D:\\Anaconda3\\envs\\lecgap\\python.exe scripts/kaggle_fine_tune.py --epochs 1
  * Kaggle GPU: python scripts/kaggle_fine_tune.py --input-dir /kaggle/input/datasets/ayushdevadiga/lecturebank \\
                   --output-dir /kaggle/working --epochs 3

CSVs expected (upload as a Kaggle dataset under your account — adjust the
input-dir if your dataset slug differs):
  prerequisite_annotation.csv :: (Source_Topic_ID, Target_Topic_ID, If_prerequisite)
  208topics.csv               :: (id, Topic, Topic_Link)

Export: the model fine-tuned on ALL data is written to <output-dir>/model/ for
download and CPU inference in the main pipeline.
"""

import argparse
import csv
import os
import sys

from collections import Counter

import numpy as np

from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from backend.pipeline.fine_tune import (
    build_train_triples,
    export_model,
    fine_tune_cross_encoder,
    predict_pairs,
)


def load_data(annot_file, topics_file):
    name_of = {}
    with open(topics_file, newline="", encoding="utf-8") as f:
        for row in csv.reader(f):
            if len(row) >= 2:
                name_of[row[0]] = row[1]
    pairs = []
    with open(annot_file, newline="", encoding="utf-8") as f:
        for src, tgt, label in csv.reader(f):
            if src in name_of and tgt in name_of:
                pairs.append((name_of[src], name_of[tgt], int(label)))
    return name_of, pairs


def run_cv(X, y, *, max_neg_ratio, epochs, batch_size, lr, device):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    metrics = []
    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y)):
        tr_idx = list(tr_idx)
        vskf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
        fit_idx, val_idx = next(iter(vskf.split(tr_idx, [y[i] for i in tr_idx])))
        fit_idx = [tr_idx[i] for i in fit_idx]
        val_idx = [tr_idx[i] for i in val_idx]

        tr_pairs = [X[i] for i in fit_idx]
        tr_labels = [y[i] for i in fit_idx]
        va_pairs = [X[i] for i in val_idx]
        va_labels = [y[i] for i in val_idx]
        te_pairs = [X[i] for i in te_idx]
        te_labels = [y[i] for i in te_idx]

        triples = build_train_triples(tr_pairs, tr_labels, max_neg_ratio=max_neg_ratio)
        model, tok = fine_tune_cross_encoder(
            triples,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            device=device,
        )

        va_logits = predict_pairs(model, tok, va_pairs)
        best = (0.0, 0.0)
        for t in np.arange(-2.0, 3.0, 0.1):
            preds = [1 if s >= t else 0 for s in va_logits]
            f = f1_score(va_labels, preds, zero_division=0)
            if f > best[0]:
                best = (f, float(t))
        thr = best[1]

        te_logits = predict_pairs(model, tok, te_pairs)
        preds = [1 if s >= thr else 0 for s in te_logits]
        m = {
            "p": precision_score(te_labels, preds, zero_division=0),
            "r": recall_score(te_labels, preds, zero_division=0),
            "f1": f1_score(te_labels, preds, zero_division=0),
            "acc": sum(p == t for p, t in zip(preds, te_labels)) / len(te_labels),
            "thr": thr,
        }
        metrics.append(m)
        print(
            f"  fold {fold+1}: P={m['p']:.3f} R={m['r']:.3f} "
            f"F1={m['f1']:.3f} acc={m['acc']:.3f} @thr={thr:.2f}",
            flush=True,
        )
    n = len(metrics)
    print("\n=== 5-fold CV (fine-tuned) summary ===")
    print(f"Precision (avg): {sum(m['p'] for m in metrics) / n:.3f}")
    print(f"Recall    (avg): {sum(m['r'] for m in metrics) / n:.3f}")
    print(f"F1        (avg): {sum(m['f1'] for m in metrics) / n:.3f}")
    print(f"Accuracy  (avg): {sum(m['acc'] for m in metrics) / n:.3f}")
    return sum(m["f1"] for m in metrics) / n


def _resolve_input_dir(input_dir: str) -> str:
    """Return a directory containing both lecturebank CSVs.

    If ``input_dir`` already holds them, use it as-is. Otherwise, on Kaggle,
    scan the standard input roots for the two CSVs (the dataset slug is not
    always predictable — e.g. /kaggle/input/datasets/<user>/<slug>/).
    """
    def present(d):
        return os.path.isfile(os.path.join(d, "prerequisite_annotation.csv")) and \
            os.path.isfile(os.path.join(d, "208topics.csv"))

    if present(input_dir):
        return input_dir

    roots = []
    if os.path.isdir("/kaggle/input"):
        roots.append("/kaggle/input")
    for root in roots:
        for dirpath, dirnames, filenames in os.walk(root):
            if "prerequisite_annotation.csv" in filenames and "208topics.csv" in filenames:
                print(f"Found lecturebank data at {dirpath}", flush=True)
                return dirpath
    return input_dir


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input-dir", default="data/lecturebank")
    ap.add_argument("--output-dir", default="data/lecturebank/model")
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--lr", type=float, default=2e-5)
    ap.add_argument("--max-neg-ratio", type=int, default=8)
    ap.add_argument("--device", default=None)
    args = ap.parse_args()

    input_dir = _resolve_input_dir(args.input_dir)
    annot = os.path.join(input_dir, "prerequisite_annotation.csv")
    topics = os.path.join(input_dir, "208topics.csv")
    name_of, pairs = load_data(annot, topics)
    X = [(a, b) for a, b, _ in pairs]
    y = [l for _, _, l in pairs]
    print(f"Loaded {len(pairs)} pairs across {len(name_of)} topics; "
          f"balance {Counter(y)}", flush=True)

    run_cv(
        X, y,
        max_neg_ratio=args.max_neg_ratio,
        epochs=args.epochs,
        batch_size=args.batch_size,
        lr=args.lr,
        device=args.device,
    )

    # Final model on ALL data for deployment.
    print("\nFine-tuning final model on all data...", flush=True)
    triples = build_train_triples(X, y, max_neg_ratio=args.max_neg_ratio)
    model, tok = fine_tune_cross_encoder(
        triples, epochs=args.epochs, batch_size=args.batch_size,
        lr=args.lr, device=args.device,
    )
    out = export_model(model, tok, args.output_dir)
    print(f"Exported fine-tuned model to {out}", flush=True)


if __name__ == "__main__":
    main()
'''
with open('/kaggle/working/kaggle_fine_tune.py', 'w') as f:
    f.write(kaggle_src)
print('wrote kaggle_fine_tune.py')

wrote kaggle_fine_tune.py


In [18]:
INPUT_DIR = '/kaggle/input/datasets/ayushdevadiga/lecturebank'

!python /kaggle/working/kaggle_fine_tune.py \
    --input-dir {INPUT_DIR} \
    --output-dir /kaggle/working/model \
    --epochs 3 --batch-size 32 --lr 2e-5 --max-neg-ratio 8

print('\nFine-tuned model is in /kaggle/working/model/ — download it for local CPU inference.')

Loaded 41921 pairs across 208 topics; balance Counter({0: 41008, 1: 913})
tokenizer_config.json: 100%|███████████████████| 350/350 [00:00<00:00, 1.68MB/s]
vocab.txt: 232kB [00:00, 62.6MB/s]
tokenizer.json: 466kB [00:00, 95.7MB/s]
special_tokens_map.json: 100%|██████████████████| 112/112 [00:00<00:00, 603kB/s]
model.safetensors: 100%|███████████████████| 90.9M/90.9M [00:01<00:00, 65.6MB/s]
Loading weights: 100%|█| 103/103 [00:00<00:00, 1469.88it/s, Materializing param=
BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstrea

In [19]:
import shutil

# Compress the folder
shutil.make_archive('model-weights-lecgap', 'zip', '/kaggle/working/model')


'/kaggle/working/model-weights-lecgap.zip'